In [1]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
from ranking_methods import rank_accuracy                                                                                   # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [2]:
previous_data = False
best_feat = None   # None = auto-pick highest |ρ| feature from corr_df
#best_feat = 'ibi_median'
#named_combo = ['ibi_median', 'hourly_entropy', 'power_8h', 'bout_len_cv']

named_combo = None
#named_combo = ['rhythm_ratio_24_over_harm', 'daily_peak_std', 'night_frac', 'num_bouts_night']
if not previous_data:
    data_folder = "./data/new_data_2"    
    individual_files = glob.glob(data_folder + "/**/*.zip", recursive=True)
    ranks = {
        "animal_1": 9,
        "animal_2": 8,
        "animal_3": 11,
        "animal_4": 10,
        "animal_5": 12,
        "animal_6": 1,
        "animal_7": 5,
        "animal_8": 6,
        "animal_9": 7,
        "animal_10": 2,
        "animal_11": 3,
        "animal_12": 4
    }
    output_folder = './results/test_results_2026_new_data'

    # Build date range between two dates (inclusive)
    start_date = date(2026, 3, 17)
    end_date   = date(2026, 3, 20)
    dates_to_keep = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                    for i in range((end_date - start_date).days + 1)]
    force_rewrite = False
    individual_files = glob.glob(data_folder + "/**/*.txt", recursive=True)

else:

    data_folder = "./data"                              
    individual_files = glob.glob(data_folder + "/unwrapped_data/**/*.txt", recursive=True)
    ranks = {
        "animal_5": 5,
        "animal_12": 8,
        "animal_9": 6,
        "animal_13": 4,
        "animal_15": 1,
        "animal_3": 3,
        "animal_8": 7,
        "animal_11": 2
    }
    output_folder = './results/test_results_2026_previous_data'


    dates_to_keep = []
    force_rewrite = False

if force_rewrite:
    for file in individual_files:
        print(file)
        folder = file.split("/")[-1].split(".zip")[0]

        
        root_folder = os.path.join(data_folder, folder)
        os.makedirs(root_folder, exist_ok=True)
        a = chr.intellicage_unwrapper([file], root_folder=root_folder, sampling_interval = '30T')
    




if dates_to_keep:
    individual_files = [f for f in individual_files if f.split("/")[-2].split(" ")[0] in dates_to_keep]


In [3]:
def get_intellicage_start_end_date(file):
    file = open(file, 'r')
    lines = file.readlines()
    lines = [line.strip() for line in lines]
    file.close()

    date_type = "%Y-%m-%d %H:%M:%S"        
    start_date = datetime.strptime(lines[1].split("Start date: ", 1)[1].strip(), date_type)
    end_date = datetime.strptime(lines[2].split("End date: ", 1)[1].strip(), date_type)
    #print(f"Start date {start_date} end date {end_date}")
    
    # Calculate number of days
    num_days = (end_date.date() - start_date.date()).days + 1
    
    return num_days

def read_data(file, labels_dict, name="", zt_0_time = None, type = 'intellicage'):

    protocol = None

    try:
        #print(f"Reading file {file}") 
        num_days = get_intellicage_start_end_date(file)
        #print(f"Number of days: {num_days}")
        labels_dict['cycle_days'] = [num_days+1]
        protocol = chr.read_protocol(name, file, zt_0_time = zt_0_time, labels_dict = labels_dict, type = type, consider_first_day = True)
    except Exception as e:
        print(f"Error reading file {file}: {e}")

    return protocol

# Split animal_3 protocol into per-day dataframes
def split_animal_protocol_by_day(animals_protocols, animal):
    animal_df = animals_protocols[animal].data.copy()
    animal_df['date'] = animal_df.index.normalize()

    # Build a dict of day -> dataframe
    animal_by_day = {
        d: animal_df[animal_df['date'] == d].drop(columns='date')
        for d in sorted(animal_df['date'].unique())
    }
    ks = list(animal_by_day.keys())
    print(f"Animal {animal} Days found:", len(ks))
    return animal_by_day


def build_animal_protocols2(
        animals_files, 
        zt_0_time=20, 
        labels_dict={'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}):

 
    animals_protocols = {}
    reads = []


    for k, v in animals_files.items():
        print(f"Animal2 {k}")
        if k not in reads:

            
                protocol = read_data(v[0], zt_0_time=zt_0_time, labels_dict=labels_dict)

                for i in range(1, len(v)):
                    try:
                        protocol.concat_protocols(read_data(v[i], zt_0_time=zt_0_time, labels_dict=labels_dict), method='sum')
                    except Exception as e:
                        print(f"Error trying to concatenate {v} with {v[i]}")


         
                protocol.resample('15T', method='sum')
                protocol.apply_filter(type = 'savgol')
                animals_protocols[f"animal_{k}"] = protocol
                reads.append(k)
 
        else:
            print(f"Animal {k} already read, skipping.")

    
    animals_by_day = {}
    for k in animals_protocols.keys():
        animals_by_day[k] = split_animal_protocol_by_day(animals_protocols, k)

    return animals_protocols, animals_by_day


#usando sum para concatenar os protocolos gera resultado melhor que o last
#Se eu quiser ler os protocolos eu preciso do zt_0_time, neste caso estou usando 20horas.
#Tambem precisariamos definir o labels dict, neste caso o que utilizariamos?
zt_0_time = 20  #Para gerar o actograma é usado o zt dado de quando a luz é acesa, que será as 20 horas. 
labels_dict = {'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}
print(ranks)
animals = [int(k.split("_")[1]) for k in ranks.keys()]
print(animals)
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols2(animals_files, zt_0_time=zt_0_time, labels_dict=labels_dict)


{'animal_1': 9, 'animal_2': 8, 'animal_3': 11, 'animal_4': 10, 'animal_5': 12, 'animal_6': 1, 'animal_7': 5, 'animal_8': 6, 'animal_9': 7, 'animal_10': 2, 'animal_11': 3, 'animal_12': 4}
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Animal 1: 4
Animal 2: 4
Animal 3: 4
Animal 4: 4
Animal 5: 4
Animal 6: 4
Animal 7: 4
Animal 8: 4
Animal 9: 4
Animal 10: 4
Animal 11: 4
Animal 12: 4
Animal2 1
Resampling data to 15T using method sum
savgol True
Animal2 2
Resampling data to 15T using method sum
savgol True
Animal2 3
Resampling data to 15T using method sum
savgol True
Animal2 4
Resampling data to 15T using method sum
savgol True
Animal2 5
Resampling data to 15T using method sum
savgol True
Animal2 6
Resampling data to 15T using method sum
savgol True
Animal2 7
Resampling data to 15T using method sum
savgol True
Animal2 8
Resampling data to 15T using method sum
savgol True
Animal2 9
Resampling data to 15T using method sum
savgol True
Animal2 10
Resampling data to 15T using method sum
savgol True
Anima

In [4]:
os.makedirs(output_folder, exist_ok=True)


output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, ranks, output_path=output_path)


output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, ranks, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

# Derived additive features

X_scaled =  get_data_scaled(all_features, feature_cols)
y = all_features['actual_rank'].values

output_correlation = f'{output_folder}/feature_correlations.csv'
corr_df = calculate_correlations(all_features, feature_cols, output_path=output_correlation)


Saving features on ./results/test_results_2026_new_data/basic_features.csv
Saving temporal features on ./results/test_results_2026_new_data/temporal_features.csv
Saving all features on ./results/test_results_2026_new_data/all_features.csv
Saving correlation on ./results/test_results_2026_new_data/feature_correlations.csv


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Diagnostic: feature heatmap sorted by true rank ─────────────────────────
# Shows z-scored feature values per animal (rows = animals sorted by rank 1→N)
# Features sorted by |ρ| with actual_rank. Ideal: a clear gradient top→bottom.

top_diag = 30   # how many top features to show
diag_feats = corr_df[corr_df.index.isin(feature_cols)].head(top_diag).index.tolist()

# Sort animals by actual rank
af_sorted = all_features.sort_values('actual_rank')
animal_labels = [f"{a} (#{int(r)})" for a, r in zip(af_sorted['animal'], af_sorted['actual_rank'])]

# Z-score matrix
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(af_sorted[diag_feats].values)

# Sign-align so that positive z → higher rank (consistent visual direction)
for j, feat in enumerate(diag_feats):
    rho_sign = np.sign(corr_df.loc[feat, 'correlation'])
    if rho_sign < 0:
        Z[:, j] *= -1

fig_heat = go.Figure(go.Heatmap(
    z=Z,
    x=diag_feats,
    y=animal_labels,
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='z (sign-aligned)'),
))
fig_heat.update_layout(
    title='Feature Heatmap — Animals sorted by Rank (top = most dominant)',
    height=max(400, 40 * len(af_sorted)),
    width=max(900, 30 * len(diag_feats)),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_heat.write_html(f"{output_folder}/feature_heatmap.html")
fig_heat.show()

# ── Scatter grid: top-6 features vs actual rank ──────────────────────────────
top6 = diag_feats[:6]
fig_scatter = make_subplots(rows=2, cols=3,
    subplot_titles=[f"{f}<br>ρ={corr_df.loc[f,'correlation']:+.3f}" for f in top6])

for idx, feat in enumerate(top6):
    r, c = divmod(idx, 3)
    x_vals = af_sorted['actual_rank'].tolist()
    y_vals = af_sorted[feat].tolist()
    fig_scatter.add_trace(go.Scatter(
        x=x_vals, y=y_vals, mode='markers+text',
        text=af_sorted['animal'].tolist(),
        textposition='top center', textfont=dict(size=7),
        marker=dict(size=9, color=x_vals, colorscale='Viridis', showscale=False),
        showlegend=False,
    ), row=r+1, col=c+1)
    fig_scatter.update_xaxes(title_text='True rank', row=r+1, col=c+1)
    fig_scatter.update_yaxes(title_text=feat, row=r+1, col=c+1)

fig_scatter.update_layout(
    title='Top Feature Scatter vs True Rank',
    height=600, width=1100,
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_scatter.write_html(f"{output_folder}/feature_scatter_vs_rank.html")
fig_scatter.show()
print(f"Saved heatmap → {output_folder}/feature_heatmap.html")
print(f"Saved scatter → {output_folder}/feature_scatter_vs_rank.html")


Saved heatmap → ./results/test_results_2026_new_data/feature_heatmap.html
Saved scatter → ./results/test_results_2026_new_data/feature_scatter_vs_rank.html


In [10]:
from util import generate_features, generate_temporal_features, combine_all_features, get_data_scaled
import pandas as pd

# ── Per-day correlation analysis ─────────────────────────────────────────────
all_days = sorted({d for animal_days in animals_by_day.values() for d in animal_days.keys()})

def _day_str(d):
    """Convert any date-like object to a plain string."""
    return pd.Timestamp(d).strftime("%Y-%m-%d")

print(f"Days found: {[_day_str(d) for d in all_days]}\n")

day_results = {}

for day in all_days:
    day_protocols = {}
    for animal, day_dict in animals_by_day.items():
        if day in day_dict:
            class _FakeProtocol:
                def __init__(self, df):
                    self.data = df
            day_protocols[animal] = _FakeProtocol(day_dict[day])

    if len(day_protocols) < 3:
        continue

    try:
        f_df = generate_features(day_protocols, ranks)
        t_df = generate_temporal_features(day_protocols, ranks)
        af_d, fc_d = combine_all_features(f_df, t_df, output_path=None)
        corr_d = calculate_correlations(af_d, fc_d)

        top5 = corr_d.head(5)
        best_rho = corr_d['correlation'].abs().max()
        best_feat_d = corr_d['correlation'].abs().idxmax()
        day_results[day] = {'best_rho': best_rho, 'best_feat': best_feat_d, 'corr_df': corr_d}
        print(f"Day {_day_str(day)}  best_rho={best_rho:.3f}  feat={best_feat_d}")
        for feat in top5.index:
            print(f"    {feat:<28s} ρ={corr_d.loc[feat,'correlation']:+.3f}  p={corr_d.loc[feat,'p_value']:.3f}")
        print()
    except Exception as e:
        print(f"Day {_day_str(day)}: error — {e}")

if day_results:
    best_day = max(day_results, key=lambda d: day_results[d]['best_rho'])
    print(f"\n→ Strongest signal day: {_day_str(best_day)}  best_rho={day_results[best_day]['best_rho']:.3f}  feat={day_results[best_day]['best_feat']}")


Days found: ['2026-03-16', '2026-03-17', '2026-03-18', '2026-03-19', '2026-03-20', '2026-03-21']

Day 2026-03-16  best_rho=0.585  feat=total_activity
    total_activity               ρ=-0.585  p=0.046
    mean_activity                ρ=-0.585  p=0.046
    activity_per_hour            ρ=-0.585  p=0.046
    night_frac                   ρ=-0.585  p=0.046
    day_minus_night              ρ=+0.585  p=0.046

Day 2026-03-17  best_rho=0.734  feat=max_median_ratio
    max_median_ratio             ρ=+0.734  p=0.007
    cosinor_acrophase            ρ=+0.657  p=0.020
    cv_activity                  ρ=+0.545  p=0.067
    median_activity              ρ=-0.539  p=0.070
    ibi_median                   ρ=+0.538  p=0.071

Day 2026-03-18  best_rho=0.808  feat=short_bout_frac
    short_bout_frac              ρ=+0.808  p=0.001
    p95_bout_len                 ρ=-0.754  p=0.005
    interdaily_stability         ρ=-0.706  p=0.010
    longest_active_run           ρ=-0.660  p=0.020
    bout_len_kurt          

In [11]:
from scipy.stats import rankdata, spearmanr
from sklearn.preprocessing import StandardScaler

# ── Per-day rank ensemble ────────────────────────────────────────────────────
# Strategy: for each valid day, compute features → find best single-feature
# rank prediction → average predicted ranks across days (Borda ensemble).
# This avoids the dilution problem caused by aggregating raw features.

good_days = [d for d in all_days if d in day_results and not np.isnan(day_results[d]['best_rho'])]
print(f"Valid days for ensemble: {[_day_str(d) for d in good_days]}")

n_animals_e = len(ranks)
animal_list_e = list(ranks.keys())  # order matches rows in per-day all_features

# Accumulate predicted rank per animal across days
rank_sum = {a: 0.0 for a in animal_list_e}
day_contributions = []

for day in good_days:
    corr_d = day_results[day]['corr_df']
    day_protocols = {}
    for animal, day_dict in animals_by_day.items():
        if day in day_dict:
            class _FP:
                def __init__(self, df): self.data = df
            day_protocols[animal] = _FP(day_dict[day])

    try:
        f_df = generate_features(day_protocols, ranks)
        t_df = generate_temporal_features(day_protocols, ranks)
        af_d, fc_d = combine_all_features(f_df, t_df, output_path=None)

        y_true_d = af_d['actual_rank'].values
        if np.any(np.isnan(y_true_d)):
            continue

        # Use top-3 features from this day's corr_df, orient by sign
        top_feats_d = corr_d[corr_d.index.isin(fc_d)].head(3).index.tolist()
        X_d = StandardScaler().fit_transform(af_d[top_feats_d].values)

        # Build composite: sign-align each feature, sum
        composite_d = np.zeros(len(af_d))
        for j, feat in enumerate(top_feats_d):
            sign_d = np.sign(corr_d.loc[feat, 'correlation'])
            composite_d += sign_d * X_d[:, j]

        # Orient composite so higher → higher rank
        rho_dir_d = spearmanr(composite_d, y_true_d).correlation
        if rho_dir_d is not None and rho_dir_d < 0:
            composite_d = -composite_d

        pred_rank_d = rankdata(composite_d, method='ordinal').astype(float)
        rho_d = spearmanr(pred_rank_d, y_true_d).correlation

        # Map back to animal names
        for i, animal in enumerate(af_d['animal'].tolist()):
            rank_sum[animal] = rank_sum.get(animal, 0.0) + pred_rank_d[i]

        day_contributions.append({
            'day': _day_str(day),
            'rho': round(float(rho_d), 3),
            'best_feat': top_feats_d[0],
            'top_feats': top_feats_d
        })
        print(f"Day {_day_str(day)}  ρ={rho_d:+.3f}  feats={top_feats_d}")

    except Exception as e:
        print(f"Day {_day_str(day)}: skipped — {e}")

# ── Average rank across days → final ensemble prediction ────────────────────
n_days_used = len(day_contributions)
print(f"\nEnsemble from {n_days_used} days")

avg_scores = {a: rank_sum[a] / n_days_used for a in animal_list_e if rank_sum.get(a, 0) > 0}
animals_ens = list(avg_scores.keys())
scores_ens  = np.array([avg_scores[a] for a in animals_ens])
true_ranks_ens = np.array([ranks[a] for a in animals_ens])

pred_ranks_ens = rankdata(scores_ens, method='ordinal').astype(int)
rho_ens, p_ens = spearmanr(pred_ranks_ens, true_ranks_ens)
mae_ens = np.mean(np.abs(pred_ranks_ens - true_ranks_ens))

print(f"\n{'─'*40}")
print(f"  Ensemble Spearman ρ : {rho_ens:+.3f}  (p={p_ens:.3f})")
print(f"  Ensemble MAE        : {mae_ens:.2f} ranks")
print(f"  Exact match         : {(np.abs(pred_ranks_ens-true_ranks_ens)==0).sum()}/{len(animals_ens)}")
print(f"  Within ±1           : {(np.abs(pred_ranks_ens-true_ranks_ens)<=1).sum()}/{len(animals_ens)}")
print(f"  Within ±2           : {(np.abs(pred_ranks_ens-true_ranks_ens)<=2).sum()}/{len(animals_ens)}")
print(f"{'─'*40}")
print(f"\n{'Animal':<12} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 34)
sorted_idx = np.argsort(true_ranks_ens)
for i in sorted_idx:
    err = abs(int(pred_ranks_ens[i]) - int(true_ranks_ens[i]))
    flag = " ✓" if err == 0 else (f" ~{err}" if err <= 2 else f" ✗{err}")
    print(f"{animals_ens[i]:<12} {int(true_ranks_ens[i]):>6} {int(pred_ranks_ens[i]):>6} {err:>6}{flag}")


Valid days for ensemble: ['2026-03-16', '2026-03-17', '2026-03-18', '2026-03-19', '2026-03-20']
Day 2026-03-16  ρ=+0.587  feats=['total_activity', 'mean_activity', 'activity_per_hour']
Day 2026-03-17  ρ=+0.608  feats=['max_median_ratio', 'cosinor_acrophase', 'cv_activity']
Day 2026-03-18  ρ=+0.811  feats=['short_bout_frac', 'p95_bout_len', 'interdaily_stability']
Day 2026-03-19  ρ=+0.909  feats=['ibi_p90', 'rolling_std_1h', 'interdaily_stability']
Day 2026-03-20  ρ=+0.804  feats=['median_bout_len', 'ibi_median', 'num_bouts_day']

Ensemble from 5 days

────────────────────────────────────────
  Ensemble Spearman ρ : +0.937  (p=0.000)
  Ensemble MAE        : 1.00 ranks
  Exact match         : 3/12
  Within ±1           : 9/12
  Within ±2           : 12/12
────────────────────────────────────────

Animal         True   Pred  Error
----------------------------------
animal_6          1      2      1 ~1
animal_10         2      4      2 ~2
animal_11         3      1      2 ~2
animal_12     

In [20]:

# ── Unsupervised ensemble using the supervised method's exact feature set ─────
#
# The supervised ensemble (cell above) already found the best features per day
# and their correct signs — stored in day_contributions + day_results[corr_df].
# 
# Idea: extract those exact (feature, sign) pairs as a validated set,
# then re-run the unsupervised prediction function using them.
# This answers: "if I calibrate once on a labelled cohort and freeze those
# features, how well does it transfer to future unlabelled cohorts?"
#
# This is the strongest possible unsupervised transfer — it uses the exact
# features the supervised method found to be best.

from scipy.stats import rankdata, spearmanr
from sklearn.preprocessing import StandardScaler

# ── Extract per-day top features + signs from the supervised run ──────────────
# day_contributions has: {'day', 'rho', 'best_feat', 'top_feats'}
# Signs come from corr_df (computed with true ranks — this is the one-time calibration)

supervised_feat_signs = {}   # feature → sign (+1 or -1), aggregated across days
supervised_feat_rhos  = {}   # feature → list of per-day |ρ| (for weighting)

for dc in day_contributions:
    day_key = next((d for d in good_days if _day_str(d) == dc['day']), None)
    if day_key is None:
        continue
    corr_d = day_results[day_key]['corr_df']
    for feat in dc['top_feats']:
        if feat in corr_d.index:
            r = float(corr_d.loc[feat, 'correlation'])
            sign = int(np.sign(r))
            # If the same feature appears on multiple days with consistent sign, keep it
            if feat not in supervised_feat_signs:
                supervised_feat_signs[feat] = sign
                supervised_feat_rhos[feat]  = [abs(r)]
            else:
                supervised_feat_rhos[feat].append(abs(r))
                # Majority sign
                supervised_feat_signs[feat] = sign  # last day wins (usually consistent)

# Build validated dict: feature → (sign, weight=mean|ρ|)
SUPERVISED_VALIDATED = {
    feat: (supervised_feat_signs[feat], float(np.mean(supervised_feat_rhos[feat])))
    for feat in supervised_feat_signs
}

print(f"Features extracted from supervised run ({len(SUPERVISED_VALIDATED)} unique features):")
print(f"  {'Feature':<35} {'weight':>7}  sign  days")
print("  " + "─" * 55)
for feat, (sign, weight) in sorted(SUPERVISED_VALIDATED.items(), key=lambda x: -x[1][1]):
    n_d = len(supervised_feat_rhos[feat])
    print(f"  {feat:<35} {weight:>7.3f}  {'+' if sign > 0 else '-'}     {n_d}d")

# ── Re-use the same prediction function (no labels at prediction time) ────────
def predict_ranks_unsupervised(animals_by_day, days, validated_feats):
    rank_sum_u = {}
    n_days_counted = 0
    for day in days:
        day_protos = {}
        for animal, day_dict_i in animals_by_day.items():
            if day in day_dict_i:
                class _FP7:
                    def __init__(self, df): self.data = df
                day_protos[animal] = _FP7(day_dict_i[day])
        if not day_protos:
            continue
        try:
            f_u  = generate_features(day_protos, ranks=None)
            t_u  = generate_temporal_features(day_protos, ranks=None)
            af_u, fc_u = combine_all_features(f_u, t_u, output_path=None)
            use_feats = [f for f in validated_feats
                         if f in af_u.columns
                         and af_u[f].notna().all()
                         and af_u[f].std() > 1e-9]
            if not use_feats:
                continue
            X_u = StandardScaler().fit_transform(af_u[use_feats].values)
            composite = sum(
                validated_feats[f][0] * validated_feats[f][1] * X_u[:, j]
                for j, f in enumerate(use_feats)
            )
            pred_day = rankdata(composite, method='ordinal').astype(float)
            for i_a, animal in enumerate(af_u['animal'].tolist()):
                rank_sum_u[animal] = rank_sum_u.get(animal, 0.0) + pred_day[i_a]
            n_days_counted += 1
        except Exception as e:
            print(f"  Day {_day_str(day)}: skipped — {e}")
    if n_days_counted == 0:
        return {}
    avg_u     = {a: rank_sum_u[a] / n_days_counted for a in rank_sum_u}
    animals_s = sorted(avg_u.keys())
    final_pred = rankdata([avg_u[a] for a in animals_s], method='ordinal').astype(int)
    return dict(zip(animals_s, final_pred))

# ── Run with supervised feature set ──────────────────────────────────────────
pred_sup_feats = predict_ranks_unsupervised(animals_by_day, good_days, SUPERVISED_VALIDATED)

# ── Three-way comparison ──────────────────────────────────────────────────────
if 'ranks' in globals() and ranks and pred_sup_feats:
    common = [a for a in pred_sup_feats if a in ranks]

    pred_sf = np.array([pred_sup_feats[a] for a in common])
    true_v  = np.array([ranks[a]          for a in common])
    rho_sf, p_sf = spearmanr(pred_sf, true_v)
    mae_sf = np.mean(np.abs(pred_sf - true_v))
    w1_sf  = int((np.abs(pred_sf - true_v) <= 1).sum())
    w2_sf  = int((np.abs(pred_sf - true_v) <= 2).sum())

    # Also recompute the mean-ρ unsupervised baseline (previous cell)
    pred_u_arr = np.array([pred_unsup[a] for a in common]) if 'pred_unsup' in globals() else None

    print(f"\n{'─'*62}")
    print(f"  {'Method':<36} {'ρ':>6}  {'MAE':>5}  {'±1':>5}  {'±2':>5}")
    print(f"{'─'*62}")
    print(f"  {'Supervised (per-day, uses true ranks)':<36} {rho_ens:>+6.3f}  {mae_ens:>5.2f}  {int((np.abs(pred_ranks_ens-true_ranks_ens)<=1).sum()):>2}/{len(true_ranks_ens)}  {int((np.abs(pred_ranks_ens-true_ranks_ens)<=2).sum()):>2}/{len(true_ranks_ens)}")
    if pred_u_arr is not None:
        rho_u2, _ = spearmanr(pred_u_arr, true_v)
        mae_u2 = np.mean(np.abs(pred_u_arr - true_v))
        w1_u2 = int((np.abs(pred_u_arr - true_v) <= 1).sum())
        w2_u2 = int((np.abs(pred_u_arr - true_v) <= 2).sum())
        print(f"  {'Unsupervised (mean-ρ features)':<36} {rho_u2:>+6.3f}  {mae_u2:>5.2f}  {w1_u2:>2}/{len(common)}  {w2_u2:>2}/{len(common)}")
    print(f"  {'Unsupervised (supervised features)':<36} {rho_sf:>+6.3f}  {mae_sf:>5.2f}  {w1_sf:>2}/{len(common)}  {w2_sf:>2}/{len(common)}")
    print(f"{'─'*62}")

    print(f"\n  {'Animal':<14} {'True':>5} {'Pred':>5} {'Err':>5}")
    print("  " + "─" * 32)
    for animal in sorted(common, key=lambda a: ranks[a]):
        err = abs(int(pred_sup_feats[animal]) - int(ranks[animal]))
        flag = " ✓" if err == 0 else (f" ~{err}" if err <= 2 else f" ✗{err}")
        print(f"  {animal:<14} {ranks[animal]:>5} {pred_sup_feats[animal]:>5} {err:>5}{flag}")


Features extracted from supervised run (14 unique features):
  Feature                              weight  sign  days
  ───────────────────────────────────────────────────────
  short_bout_frac                       0.808  +     1d
  median_bout_len                       0.771  +     1d
  ibi_p90                               0.760  -     1d
  rolling_std_1h                        0.755  +     1d
  p95_bout_len                          0.754  -     1d
  ibi_median                            0.740  +     1d
  max_median_ratio                      0.734  +     1d
  interdaily_stability                  0.717  -     2d
  num_bouts_day                         0.709  -     1d
  cosinor_acrophase                     0.657  +     1d
  total_activity                        0.585  -     1d
  mean_activity                         0.585  -     1d
  activity_per_hour                     0.585  -     1d
  cv_activity                           0.545  +     1d

──────────────────────────────────────

In [6]:

# ── Build TOP_FEATURES dynamically from corr_df (cell 4 output) ─────────────
min_rho  = 0.20   # lowered threshold — cast wider net, search will still rank by |ρ|
top_n    = 30     # more candidates

top_corr = corr_df[corr_df.index.isin(feature_cols)].copy()
top_corr = top_corr[top_corr['correlation'].abs() >= min_rho].head(top_n)

TOP_FEATURES = {
    feat: (int(np.sign(row['correlation'])), round(abs(row['correlation']), 3))
    for feat, row in top_corr.iterrows()
}

print(f"TOP_FEATURES built from corr_df ({len(TOP_FEATURES)} features, |ρ| >= {min_rho}):")
print(f"{'Feature':<25} {'sign':>5}  {'|ρ|':>6}  {'p':>7}")
print("-" * 55)
for feat, (sign, rho_abs) in TOP_FEATURES.items():
    p = corr_df.loc[feat, 'p_value']
    sig = "**" if p < 0.01 else " *" if p < 0.05 else " ~" if p < 0.1 else ""
    print(f"  {feat:<23} {sign:>+5}  {rho_abs:>6.3f}  {p:>7.3f} {sig}")

# ── Exhaustive search ────────────────────────────────────────────────────────
available_feats = {f: v for f, v in TOP_FEATURES.items() if f in feature_cols}
feat_names = list(available_feats.keys())
y_true_cmp = all_features['actual_rank'].values

feat_vectors = {}
for feat, (sign, weight) in available_feats.items():
    col_idx = feature_cols.index(feat)
    feat_vectors[feat] = sign * X_scaled[:, col_idx]

best_results = []
total_combos = sum(len(list(combinations(feat_names, k))) for k in range(1, 6)) * 2

for k in range(1, 6):
    for combo in combinations(feat_names, k):
        composite_c = sum(feat_vectors[f] for f in combo)
        for polarity in [1, -1]:
            pred = rankdata(-polarity * composite_c).astype(int)
            rho_c, p_c = spearmanr(y_true_cmp, pred)
            mae_c = np.abs(y_true_cmp - pred).mean()
            if rho_c > 0:
                best_results.append((rho_c, p_c, mae_c, k, combo, polarity))

best_results.sort(key=lambda x: (-x[0], x[2]))

print(f"\nTop 20 feature combinations (out of {len(best_results)} valid, {total_combos} tested):\n")
print(f"{'ρ':>6} {'p':>7} {'MAE':>5}  {'k':>2}  Features")
print("-" * 90)
for rho_c, p_c, mae_c, k, combo, pol in best_results[:20]:
    sig = "**" if p_c < 0.01 else " *" if p_c < 0.05 else " ~" if p_c < 0.1 else "  "
    print(f"{rho_c:+.3f} {p_c:7.3f} {mae_c:5.2f}  {k:2d}  {', '.join(combo)} {sig}")

# ── Feature vote across top-10 results ──────────────────────────────────────
top_feats_pool = {}
for rho_c, p_c, mae_c, k, combo, pol in best_results[:10]:
    for f in combo:
        top_feats_pool[f] = top_feats_pool.get(f, 0) + rho_c

top_feats_ranked = sorted(top_feats_pool.items(), key=lambda x: -x[1])
print(f"\nMost frequent features in top-10 combos (by accumulated ρ):")
for feat, score in top_feats_ranked:
    sign, _ = available_feats[feat]
    print(f"  {feat:<25s}  score={score:.3f}  sign={'↑' if sign>0 else '↓'}")

# ── Apply the best combination ──────────────────────────────────────────────
best_rho, best_p, best_mae, best_k, best_combo, best_pol = best_results[0]
best_combo_str = " + ".join(best_combo)

print(f"\n→ Best combo (k={best_k}): {list(best_combo)}")
print(f"  ρ={best_rho:+.3f}, p={best_p:.3f}, MAE={best_mae:.2f}")

composite_best = best_pol * sum(feat_vectors[f] for f in best_combo)
pred_best = rankdata(-composite_best).astype(int)
all_features['predicted_rank_best'] = pred_best
all_features['composite_score_best'] = composite_best

comparison_best = all_features.set_index('animal')[['predicted_rank_best', 'composite_score_best']].copy()
comparison_best['true_rank'] = pd.Series(ranks)
comparison_best = comparison_best.sort_values('true_rank')
comparison_best['error'] = (comparison_best['predicted_rank_best'] - comparison_best['true_rank']).abs()

n = len(comparison_best)
print(f"\n  Spearman ρ      : {best_rho:+.3f}  (p={best_p:.3f})")
print(f"  MAE             : {best_mae:.2f} ranks")
print(f"  Exact match     : {(comparison_best['error']==0).sum()}/{n}  ({100*(comparison_best['error']==0).mean():.0f}%)")
print(f"  Within ±1 rank  : {(comparison_best['error']<=1).sum()}/{n}  ({100*(comparison_best['error']<=1).mean():.0f}%)")
print(f"  Within ±2 ranks : {(comparison_best['error']<=2).sum()}/{n}  ({100*(comparison_best['error']<=2).mean():.0f}%)")
print(f"\n{'Animal':<12} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 34)
for animal, row in comparison_best.iterrows():
    flag = " ✓" if row['error'] == 0 else (f" ~{int(row['error'])}" if row['error'] <= 2 else f" ✗{int(row['error'])}")
    print(f"{animal:<12} {int(row['true_rank']):>6} {int(row['predicted_rank_best']):>6} {int(row['error']):>6}{flag}")

# ── Save ranked output sorted by predicted rank ──────────────────────────────
ranked_output = comparison_best.copy().reset_index()
ranked_output = ranked_output.rename(columns={
    'predicted_rank_best': 'predicted_rank',
    'composite_score_best': 'composite_score',
    'error': 'absolute_error',
})
ranked_output['features_used'] = best_combo_str
ranked_output['spearman_rho']  = round(best_rho, 4)
ranked_output['p_value']       = round(best_p, 4)
ranked_output['mae']           = round(best_mae, 4)
ranked_output = ranked_output.sort_values('predicted_rank')

ranked_output_path = f"{output_folder}/best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)

print(f"\n── Best combo ranked output (sorted by predicted rank) ──")
print(ranked_output[['animal', 'predicted_rank', 'true_rank', 'absolute_error', 'composite_score']].to_string(index=False))
print(f"\nSaved to {ranked_output_path}")


TOP_FEATURES built from corr_df (30 features, |ρ| >= 0.2):
Feature                    sign     |ρ|        p
-------------------------------------------------------
  rhythm_ratio_24_over_harm    +1   0.790    0.002 **
  power_24h                  +1   0.706    0.010  *
  autocorr_12h               -1   0.664    0.018  *
  power_12h                  -1   0.657    0.020  *
  cosinor_amplitude          +1   0.657    0.020  *
  median_activity            -1   0.647    0.023  *
  activity_kurtosis          -1   0.608    0.036  *
  high_activity_frac         +1   0.578    0.049  *
  activity_skew              -1   0.559    0.059  ~
  bout_len_cv                -1   0.545    0.067  ~
  max_activity               -1   0.545    0.067  ~
  peak_to_mean_ratio         -1   0.524    0.080  ~
  max_median_ratio           +1   0.517    0.085  ~
  cosinor_mesor              +1   0.517    0.085  ~
  night_activity             -1   0.476    0.118 
  night_mean_activity        -1   0.476    0.118 
  roll

In [7]:
# k            : number of top features used by top-k methods
# best_feat_idx: feature name (or int index) for the single best feature proxy
# named_combo  : list of feature names for the exhaustive-search best combo

print(best_combo)
best_feat = None
if best_feat is None:
    print("Best feat is none, computing")
    best_feat = corr_df[corr_df.index.isin(feature_cols)]['correlation'].abs().idxmax()

if named_combo is None:
    print("Named combo is none, using computed")
    named_combo = list(best_combo) if 'best_combo' in dir() else None
    
print(best_feat)
print(named_combo)

#best_feat = 'power_8h'

feature_rhos, proxies_raw = build_all_proxies(
    all_features, feature_cols, X_scaled,
    k=3, best_feat_idx=best_feat,
    named_combo=named_combo
)





print(f"Best single feature: {best_feat}")
print('Top feature correlations (Spearman):')
print(feature_rhos.head(10))

proxy_output_path = f"{output_folder}/proxy_summary.csv"
proxy_rows, proxy_summary = evaluate_proxies(proxies_raw, all_features, verbose=True, output_path=proxy_output_path)

print("\nUnsupervised proxies vs rank (Spearman rho, MAE):")
for row in proxy_rows:
    print(f"{row['proxy']:<40s} rho={row['rho']: .3f}, MAE={row['mae']: .2f}, tau={row['kendall_tau']: .3f}")


('power_24h', 'high_activity_frac', 'activity_per_bout')
Best feat is none, computing
Named combo is none, using computed
rhythm_ratio_24_over_harm
['power_24h', 'high_activity_frac', 'activity_per_bout']
Best single feature: rhythm_ratio_24_over_harm
Top feature correlations (Spearman):
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
median_activity             -0.646554
activity_kurtosis           -0.608392
high_activity_frac           0.577552
activity_skew               -0.559441
bout_len_cv                 -0.545455
dtype: float64
Score [0.17289371 0.24268741 0.33396411 0.2049167  0.25076042 0.09744994
 0.15736368 0.3737061  0.16437793 0.04575505 0.10947877 0.06043491] scores oriented [0.17289371 0.24268741 0.33396411 0.2049167  0.25076042 0.09744994
 0.15736368 0.3737061  0.16437793 0.04575505 0.10947877 0.06043491]

Best feature (rhythm_ra

In [8]:
actual_ranks = all_features['actual_rank'].values

for name,score in proxies_raw.items():
    print(score)
    ord_idx = np.argsort(score)
    print(ord_idx)
    ordered_animals = all_features.iloc[ord_idx]['animal'].tolist()
    print(name)
    pred_rank = rankdata(score, method='ordinal')
    print(ordered_animals)
    print(pred_rank)
    print(actual_ranks)

    metrics = rank_accuracy(actual_ranks, pred_rank)
    print(metrics)
    print()


[0.17289371 0.24268741 0.33396411 0.2049167  0.25076042 0.09744994
 0.15736368 0.3737061  0.16437793 0.04575505 0.10947877 0.06043491]
[ 9 11  5 10  6  8  0  3  1  4  2  7]
Best feature (rhythm_ratio_24_over_harm)
['animal_10', 'animal_12', 'animal_6', 'animal_11', 'animal_7', 'animal_9', 'animal_1', 'animal_4', 'animal_2', 'animal_5', 'animal_3', 'animal_8']
[ 7  9 11  8 10  3  5 12  6  1  4  2]
[ 9  8 11 10 12  1  5  6  7  2  3  4]
{'accuracy': 0.16666666666666666, 'within_1': 0.5, 'within_2': 0.9166666666666666, 'mae': 1.6666666666666667, 'rho': 0.7902097902097903}

[ 1.02421588  0.7916306   3.66128009  1.49751784  1.84938749 -2.20416197
 -1.31586378  0.78188016 -0.26620678 -2.15716699 -2.034411   -1.62810154]
[ 5  9 10 11  6  8  7  1  0  3  4  2]
Best combo (power_24h + high_activity_frac + activity_per_bout)
['animal_6', 'animal_10', 'animal_11', 'animal_12', 'animal_7', 'animal_9', 'animal_8', 'animal_2', 'animal_1', 'animal_4', 'animal_5', 'animal_3']
[ 9  8 12 10 11  1  5  7  6

##### Total activy per day tem a soma de todos os dias. Ë possivel ver isso claramente abrindo o arquivo collect data onde será visto a atividade por zt de 0.5 em 0.5, somando-se todos os valores da coluna o valor para cada dia será igual ao total day activity

In [163]:
output_corr = f"{output_folder}/feature_correlations.html"
fig_corr = plot_correlation(corr_df, output_corr)

male_comparison_output = f"{output_folder}/animal_comparison_all.html"
fig_activity = plot_animals_activity(animals_protocols, ranks, output_path=male_comparison_output)

output_path = f"{output_folder}/cross_correlation_matrix.html"
corr_sync, pval_sync, fig_cross = plot_cross_correlation(animals_protocols, ranks, output_path=output_path)

animal_1 in ranks, adding to plot
animal_2 in ranks, adding to plot
animal_3 in ranks, adding to plot
animal_4 in ranks, adding to plot
animal_5 in ranks, adding to plot
animal_6 in ranks, adding to plot
animal_7 in ranks, adding to plot
animal_8 in ranks, adding to plot
animal_9 in ranks, adding to plot
animal_10 in ranks, adding to plot
animal_11 in ranks, adding to plot
animal_12 in ranks, adding to plot



Average Activity Synchronization:
       animal  avg_correlation  actual_rank
5    animal_6         0.669877            1
9   animal_10         0.738562            2
10  animal_11         0.747010            3
11  animal_12         0.722566            4
6    animal_7         0.747960            5
7    animal_8         0.704673            6
8    animal_9         0.751670            7
1    animal_2         0.643215            8
0    animal_1         0.718421            9
3    animal_4         0.724619           10
2    animal_3         0.630318           11
4    animal_5         0.627037           12

Correlation between synchronization and rank: -0.462 (p=0.131)


In [164]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# ── Build per-proxy result dataframes from proxy_rows + proxies_raw ─────────
# Replicate evaluate_proxies orientation + eval_ordering exactly:
#   1. orient scores so rho(scores, actual_rank) >= 0
#      → higher score = higher rank number = LESS dominant
#   2. rankdata(scores_oriented, method='ordinal')
#      → rank 1 = lowest score = MOST dominant ✓
actual_ranks = all_features['actual_rank'].values
animal_names = all_features['animal'].tolist()

# Sort index by true rank so scatter x-axis goes 1 → N
sort_idx = np.argsort(actual_ranks)
actual_ranks_sorted = actual_ranks[sort_idx]
animal_names_sorted = [animal_names[i] for i in sort_idx]

proxy_results = {}
for row in proxy_rows:
    name   = row['proxy']
    scores = proxies_raw[name]
    rho_dir = spearmanr(scores, actual_ranks).correlation

    # Mirror evaluate_proxies: orient so rho >= 0
    scores_oriented = scores if (rho_dir is None or rho_dir >= 0) else -scores
    # Mirror eval_ordering: rankdata with ordinal (no negation)
    pred_ranks = rankdata(scores_oriented, method='ordinal').astype(int)

    # Sort by true rank for the scatter plot
    pred_ranks_sorted = pred_ranks[sort_idx]
    errors_sorted     = np.abs(pred_ranks_sorted - actual_ranks_sorted)

    proxy_results[name] = {
        'animals':   animal_names_sorted,
        'true_rank': actual_ranks_sorted.tolist(),
        'pred_rank': pred_ranks_sorted.tolist(),
        'errors':    errors_sorted.tolist(),
        'rho':       row['rho'],
        'mae':       row['mae'],
        'within_1':  row['within_1'],
        'n':         len(actual_ranks),
    }

n_methods  = len(proxy_results)
n_animals  = len(animal_names)
diag_vals  = list(range(1, n_animals + 2))

# Sort methods by |ρ| descending so the best appear first
sorted_names = sorted(proxy_results.keys(),
                      key=lambda n: -abs(proxy_results[n]['rho']))

# ── Figure 1: Summary overview (ρ and MAE for all methods) ──────────────────
summary_df = pd.DataFrame(proxy_rows).sort_values('rho', key=abs, ascending=True)

fig_summary = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Spearman ρ with Actual Rank', 'Mean Absolute Error (MAE)'),
    horizontal_spacing=0.18
)

colors_rho = ['#2E86AB' if r >= 0 else '#E84855' for r in summary_df['rho']]

fig_summary.add_trace(go.Bar(
    x=summary_df['rho'], y=summary_df['proxy'], orientation='h',
    marker_color=colors_rho,
    text=[f"{r:+.3f}" for r in summary_df['rho']],
    textposition='outside',
    showlegend=False,
), row=1, col=1)

fig_summary.add_trace(go.Bar(
    x=summary_df['mae'], y=summary_df['proxy'], orientation='h',
    marker_color='#F18F01',
    text=[f"{m:.2f}" for m in summary_df['mae']],
    textposition='outside',
    showlegend=False,
), row=1, col=2)

fig_summary.add_vline(x=0, line_color='black', line_width=1, row=1, col=1)

fig_summary.update_layout(
    title=dict(text='Proxy Methods — Summary (ρ & MAE)', x=0.5),
    height=max(350, 38 * n_methods),
    width=1200,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=280),
)
fig_summary.update_xaxes(title_text='Spearman ρ', row=1, col=1, range=[-1.05, 1.05])
fig_summary.update_xaxes(title_text='MAE (ranks)', row=1, col=2)

fig_summary.write_html(f"{output_folder}/rank_evaluation_summary.html")
fig_summary.show()
print(f"Saved → {output_folder}/rank_evaluation_summary.html")

# ── Figure 2: Per-method grid — 2 methods per row, each with scatter + bar ──
METHODS_PER_ROW = 2
n_rows = math.ceil(n_methods / METHODS_PER_ROW)
v_spacing = min(0.06, round(0.9 / max(n_rows - 1, 1), 4)) if n_rows > 1 else 0.05

subplot_titles = []
padded = sorted_names + [None] * (n_rows * METHODS_PER_ROW - n_methods)

for i in range(n_rows):
    for j in range(METHODS_PER_ROW):
        idx = i * METHODS_PER_ROW + j
        name = padded[idx]
        if name is not None:
            r = proxy_results[name]
            subplot_titles += [
                f"{name}<br><sup>ρ={r['rho']:+.3f}, MAE={r['mae']:.2f}, W±1={r['within_1']*100:.0f}%</sup>",
                "Per-Animal Error",
            ]
        else:
            subplot_titles += ["", ""]

fig_grid = make_subplots(
    rows=n_rows, cols=4,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.07,
    vertical_spacing=v_spacing,
    column_widths=[0.3, 0.2, 0.3, 0.2],
)

for i, name in enumerate(sorted_names):
    r           = proxy_results[name]
    grid_row    = i // METHODS_PER_ROW + 1
    pair_idx    = i %  METHODS_PER_ROW
    scatter_col = 1 + pair_idx * 2
    bar_col     = scatter_col + 1

    true_r  = r['true_rank']
    pred_r  = r['pred_rank']
    errors  = r['errors']
    animals = r['animals']

    err_colors = ['#2ecc71' if e == 0 else '#f39c12' if e <= 2 else '#e74c3c'
                  for e in errors]

    # Scatter: x=True Rank, y=Predicted Rank — perfect prediction lies on diagonal
    fig_grid.add_trace(go.Scatter(
        x=true_r, y=pred_r,
        mode='markers+text',
        text=animals,
        textposition='top center',
        textfont=dict(size=7),
        marker=dict(color=err_colors, size=10, line=dict(color='white', width=1)),
        showlegend=False,
    ), row=grid_row, col=scatter_col)

    fig_grid.add_trace(go.Scatter(
        x=diag_vals, y=diag_vals, mode='lines',
        line=dict(color='gray', dash='dash', width=1),
        showlegend=False,
    ), row=grid_row, col=scatter_col)

    fig_grid.add_trace(go.Scatter(
        x=diag_vals, y=[v + 1 for v in diag_vals], mode='lines',
        line=dict(color='lightgray', dash='dot', width=1), showlegend=False,
    ), row=grid_row, col=scatter_col)
    fig_grid.add_trace(go.Scatter(
        x=diag_vals, y=[v - 1 for v in diag_vals], mode='lines',
        line=dict(color='lightgray', dash='dot', width=1), showlegend=False,
    ), row=grid_row, col=scatter_col)

    fig_grid.update_xaxes(title_text="True Rank", row=grid_row, col=scatter_col)
    fig_grid.update_yaxes(title_text="Pred Rank", row=grid_row, col=scatter_col)

    # Bar: per-animal error sorted by true rank (rank 1 = most dominant on left)
    fig_grid.add_trace(go.Bar(
        x=animals, y=errors,
        marker_color=err_colors,
        text=[str(int(e)) for e in errors],
        textposition='outside',
        showlegend=False,
    ), row=grid_row, col=bar_col)

    fig_grid.update_xaxes(tickangle=-45, tickfont=dict(size=7), row=grid_row, col=bar_col)
    fig_grid.update_yaxes(title_text="Error", row=grid_row, col=bar_col)

fig_grid.update_layout(
    title=dict(text='Rank Evaluation — All Proxy Methods', x=0.5),
    height=350 * n_rows,
    width=1300,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
)

fig_grid.write_html(f"{output_folder}/rank_evaluation_all_methods.html")
fig_grid.show()
print(f"Saved → {output_folder}/rank_evaluation_all_methods.html")


Saved → ./results/test_results_2026_new_data/rank_evaluation_summary.html


Saved → ./results/test_results_2026_new_data/rank_evaluation_all_methods.html
